In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
BASE_DIR = "/content/drive/MyDrive/"

train_df = pd.read_csv(f"{BASE_DIR}/weighted_fusion_results/fusion_train.csv")
val_df   = pd.read_csv(f"{BASE_DIR}/weighted_fusion_results/fusion_val.csv")
test_df  = pd.read_csv(f"{BASE_DIR}/weighted_fusion_results/fusion_test.csv")

# ONLY PROBABILITIES (NO MODALITY)
features = ["DR_prob", "Gl_prob", "AMD_prob", "DED_prob"]

targets = ["DR", "Glaucoma", "AMD", "DED"]

X_train = train_df[features].values
y_train = train_df[targets].values

X_val = val_df[features].values
y_val = val_df[targets].values

X_test = test_df[features].values
y_test = test_df[targets].values

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

In [ ]:
class FusionDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
train_loader = DataLoader(FusionDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader   = DataLoader(FusionDataset(X_val, y_val), batch_size=32, shuffle=False)
test_loader  = DataLoader(FusionDataset(X_test, y_test), batch_size=32, shuffle=False)

In [ ]:
class AttentionNoModality(nn.Module):
    def __init__(self, input_dim=4):
        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(16, input_dim),
            nn.Sigmoid()
        )

        self.classifier = nn.Sequential(
            nn.Linear(4, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 4)
        )

    def forward(self, x):
        attn = self.attention(x)
        weighted = x * (1 + attn)
        out = self.classifier(weighted)
        return out

In [ ]:
model = AttentionNoModality().to(device)

pos_weights = torch.tensor([2.8, 1.5, 1.0, 1.0]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

In [ ]:
SAVE_DIR = os.path.join(BASE_DIR, "attention_no_modality_results")
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
epochs = 25
best_val_f1 = 0

for epoch in range(epochs):
    model.train()
    train_loss = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # VALIDATION
    model.eval()
    val_preds, val_targets = [], []

    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)

            out = model(xb)
            probs = torch.sigmoid(out)

            val_preds.append(probs.cpu().numpy())
            val_targets.append(yb.cpu().numpy())

    val_preds = np.vstack(val_preds)
    val_targets = np.vstack(val_targets)

    val_bin = (val_preds > 0.5).astype(int)
    val_f1 = f1_score(val_targets, val_bin, average="macro")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "ablation_model.pth"))

    print(f"Epoch {epoch+1}/{epochs} | Loss: {train_loss:.4f} | Val F1: {val_f1:.4f}")

Epoch 1/25 | Loss: 272.6444 | Val F1: 0.6064
Epoch 2/25 | Loss: 87.4485 | Val F1: 0.8527
Epoch 3/25 | Loss: 74.3055 | Val F1: 0.8533
Epoch 4/25 | Loss: 71.4458 | Val F1: 0.8590
Epoch 5/25 | Loss: 69.3676 | Val F1: 0.8590
Epoch 6/25 | Loss: 67.0079 | Val F1: 0.8579
Epoch 7/25 | Loss: 66.4555 | Val F1: 0.8650
Epoch 8/25 | Loss: 66.8439 | Val F1: 0.8642
Epoch 9/25 | Loss: 65.7232 | Val F1: 0.8640
Epoch 10/25 | Loss: 64.8653 | Val F1: 0.8642
Epoch 11/25 | Loss: 64.3533 | Val F1: 0.8662
Epoch 12/25 | Loss: 63.0529 | Val F1: 0.8684
Epoch 13/25 | Loss: 62.9751 | Val F1: 0.8702
Epoch 14/25 | Loss: 62.6087 | Val F1: 0.8675
Epoch 15/25 | Loss: 61.7452 | Val F1: 0.8723
Epoch 16/25 | Loss: 61.4985 | Val F1: 0.8770
Epoch 17/25 | Loss: 61.2325 | Val F1: 0.8770
Epoch 18/25 | Loss: 61.1197 | Val F1: 0.8777
Epoch 19/25 | Loss: 60.8326 | Val F1: 0.8752
Epoch 20/25 | Loss: 60.5099 | Val F1: 0.8802
Epoch 21/25 | Loss: 59.8218 | Val F1: 0.8771
Epoch 22/25 | Loss: 59.3555 | Val F1: 0.8763
Epoch 23/25 | Loss

In [ ]:
model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "ablation_model.pth")))
model.eval()

AttentionNoModality(
  (attention): Sequential(
    (0): Linear(in_features=4, out_features=16, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=16, out_features=4, bias=True)
    (4): Sigmoid()
  )
  (classifier): Sequential(
    (0): Linear(in_features=4, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=32, out_features=16, bias=True)
    (4): ReLU()
    (5): Linear(in_features=16, out_features=4, bias=True)
  )
)

In [ ]:
thresholds = []

threshold_ranges = [
    np.linspace(0.6, 0.72, 200),   # DR
    np.linspace(0.55, 0.7, 200),   # Glaucoma
    np.linspace(0.1, 0.3, 100),    # AMD
    np.linspace(0.4, 0.6, 100)     # DED
]

for i in range(4):
    best_t, best_f1 = 0.5, 0

    for t in threshold_ranges[i]:
        preds_bin = (val_preds[:, i] > t).astype(int)
        f1 = f1_score(val_targets[:, i], preds_bin)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    thresholds.append(best_t)

print("Optimal thresholds:", thresholds)

Optimal thresholds: [np.float64(0.6138693467336683), np.float64(0.55), np.float64(0.10404040404040404), np.float64(0.4)]


In [ ]:
test_preds, test_targets = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)

        out = model(xb)
        probs = torch.sigmoid(out)

        test_preds.append(probs.cpu().numpy())
        test_targets.append(yb.cpu().numpy())

test_preds = np.vstack(test_preds)
test_targets = np.vstack(test_targets)

In [ ]:
final_preds = np.zeros_like(test_preds)

for i in range(4):
    final_preds[:, i] = (test_preds[:, i] > thresholds[i]).astype(int)

In [ ]:
diseases = ["DR", "Glaucoma", "AMD", "DED"]

results = []

for i, disease in enumerate(diseases):
    auc = roc_auc_score(test_targets[:, i], test_preds[:, i])
    acc = accuracy_score(test_targets[:, i], final_preds[:, i])
    prec = precision_score(test_targets[:, i], final_preds[:, i], zero_division=0)
    rec = recall_score(test_targets[:, i], final_preds[:, i])
    f1 = f1_score(test_targets[:, i], final_preds[:, i])

    results.append([disease, auc, acc, prec, rec, f1])

results_df = pd.DataFrame(results, columns=["Disease", "AUC", "Accuracy", "Precision", "Recall", "F1"])
print(results_df)

    Disease       AUC  Accuracy  Precision    Recall        F1
0        DR  0.986527  0.960837   0.636605  0.860215  0.731707
1  Glaucoma  0.976560  0.937695   0.846348  0.809639  0.827586
2       AMD  0.999369  0.995327   0.990955  0.987976  0.989463
3       DED  0.999996  0.999555   0.965517  1.000000  0.982456


In [ ]:
results_df.to_csv(os.path.join(SAVE_DIR, "metrics.csv"), index=False)
np.save(os.path.join(SAVE_DIR, "thresholds.npy"), thresholds)
torch.save(model.state_dict(), os.path.join(SAVE_DIR, "ablation_model.pth"))

In [ ]:
# ===== FIXED THRESHOLD (0.5) =====

fixed_preds = (test_preds > 0.5).astype(int)

results_fixed = []

for i, disease in enumerate(["DR", "Glaucoma", "AMD", "DED"]):
    auc = roc_auc_score(test_targets[:, i], test_preds[:, i])
    acc = accuracy_score(test_targets[:, i], fixed_preds[:, i])
    prec = precision_score(test_targets[:, i], fixed_preds[:, i], zero_division=0)
    rec = recall_score(test_targets[:, i], fixed_preds[:, i])
    f1 = f1_score(test_targets[:, i], fixed_preds[:, i])

    results_fixed.append([disease, auc, acc, prec, rec, f1])

results_fixed_df = pd.DataFrame(
    results_fixed,
    columns=["Disease", "AUC", "Accuracy", "Precision", "Recall", "F1"]
)

print("=== Fixed Threshold (0.5) ===")
print(results_fixed_df)

=== Fixed Threshold (0.5) ===
    Disease       AUC  Accuracy  Precision    Recall        F1
0        DR  0.986527  0.954384   0.584091  0.921147  0.714882
1  Glaucoma  0.976560  0.937027   0.829916  0.828916  0.829415
2       AMD  0.999369  0.995105   0.993927  0.983968  0.988922
3       DED  0.999996  0.999555   0.965517  1.000000  0.982456


In [ ]:
# ===== SAVE BOTH RESULTS =====

results_fixed_df.to_csv(os.path.join(SAVE_DIR, "metrics_fixed.csv"), index=False)
results_df.to_csv(os.path.join(SAVE_DIR, "metrics_optimized.csv"), index=False)

np.save(os.path.join(SAVE_DIR, "thresholds.npy"), thresholds)

print("All results saved.")

All results saved.
